In [35]:
import pandas as pd
import numpy as np
import os
import re
import json as js
from pathlib import Path
from tqdm import tqdm
import shutil

In [36]:
directory_JSON_raw = "../[Full] Bentuk JSON"
directory_hasil = "../[Full] Raw CSV JSON all information"
shutil.rmtree(directory_hasil)
os.makedirs(directory_hasil)

list_json_raw = os.listdir(directory_JSON_raw)

### Algoritma Ekstraksi Informasi JSON ###

Note : 1 page terdiri dari beberapa block, 1 block terdiri dari beberapa line, 1 line terdiri dari beberapa spans, dan 1 span terdiri dari beberapa detail

In [37]:
def extract_information_json(data):
    result = {
        "text":[],
        "size":[],
        "font":[],
        "x0":[], #[0]
        "y0":[], #[1]
        "x1":[], #[2]
        "y1":[], #[3]
        "page":[]
    }
    
    lines = []
    line_pages = []
    for key in data:
        page = data.get(key)
        for block in page.get("text").get("blocks"):
            if block.get("lines") is not None:
                lines.append(block.get("lines"))
                line_pages.append(page.get("page")) 
    
    spans = []
    span_pages = []
    for i in range(len(lines)):
        ln = lines[i]
        for detail_ln in ln:
            if detail_ln.get("spans") is not None:
                spans.append(detail_ln.get("spans"))
                span_pages.append(line_pages[i])
    
    for i in range(len(spans)):
        span = spans[i]
        for detail in span:
            result.get("text").append(detail.get("text"))
            result.get("size").append(detail.get("size"))
            result.get("font").append(detail.get("font"))
            result.get("x0").append(round(detail.get("bbox")[0],2))
            result.get("y0").append(round(detail.get("bbox")[1],2))
            result.get("x1").append(round(detail.get("bbox")[2],2))
            result.get("y1").append(round(detail.get("bbox")[3],2))
            result.get("page").append(span_pages[i]+1)
    
    return result

In [38]:
def is_contain_only_whitespaces(s):
    if re.match(r'^\s*$', str(s)): return True
    return False

def clean_information(data):
    result = {
        "text":[],
        "size":[],
        "font":[],
        "x0":[], #[0]
        "y0":[], #[1]
        "x1":[], #[2]
        "y1":[], #[3]
        "page":[]
    }
    
    i = 0
    while i < len(data["text"]):
        text = data["text"][i]
        if is_contain_only_whitespaces(text): # skip kata yang hanya mengandung whitespace
            i += 1
        else:
            result["text"].append(data["text"][i].strip())
            result["size"].append(data["size"][i])
            result["font"].append(data["font"][i])
            result["x0"].append(data["x0"][i])
            result["y0"].append(data["y0"][i])
            result["x1"].append(data["x1"][i])
            result["y1"].append(data["y1"][i])
            result["page"].append(data["page"][i])
            i += 1
            
    return result

In [39]:
def seperate_span(data): # tokenisasi span berdasarkan spasi
    result = {
        "text":[],
        "size":[],
        "font":[],
        "x0":[], #[0]
        "y0":[], #[1]
        "x1":[], #[2]
        "y1":[], #[3]
        "page":[]
    }
    
    i = 0
    while i < len(data["text"]):
        curr_font = data["font"][i].lower()
        list_txt = data["text"][i].strip().split(" ")
        
        selisih_x = data["x1"][i] - data["x0"][i] # x1 - x0
        satuan_x = selisih_x/len(data["text"][i])
                
        if len(list_txt)==1: # jika span ternyata hanya satu kata
            result["text"].append(data["text"][i].strip())
            result["size"].append(data["size"][i])
            result["font"].append(data["font"][i])
            result["x0"].append(data["x0"][i])
            result["y0"].append(data["y0"][i])
            result["x1"].append(data["x1"][i])
            result["y1"].append(data["y1"][i])
            result["page"].append(data["page"][i])
            i+=1
        else:
            x0 = data["x0"][i]
            x1 = data["x1"][i] - len(data["text"][i])*satuan_x
            x1 = round(x1,2)

            y0 = data["y0"][i] # koordinat y tetap
            y1 = data["y1"][i]

            for txt in list_txt:
                x1 += len(txt) * satuan_x
                x1 = round(x1,2)

                result["text"].append(txt.strip())
                result["size"].append(data["size"][i])
                result["font"].append(data["font"][i])
                result["x0"].append(x0)
                result["y0"].append(y0)
                result["x1"].append(x1)
                result["y1"].append(y1)
                result["page"].append(data["page"][i])

                x0 += (len(txt)+1) * satuan_x
                x0 = round(x0,2)
            i+=1
            
        
    return result

In [40]:
def is_contain_bold_and_italic(font):
    contains_bold = False; contains_italic = False
    for i in font:
        if "bold" in i.lower(): contains_bold = True
        if "italic" in i.lower(): contains_italic = True
        if contains_bold == True and contains_italic == True: return True
    return False

def is_contain_bold(font):
    contains_bold = False
    for i in font:
        if "bold" in i.lower(): contains_bold = True
    return contains_bold

def count_bold(font):
    cnt = 0
    for i in font:
        if "bold" in i.lower(): 
            cnt += 1
    return cnt

In [41]:
daftar_kamus_layak = []
daftar_kamus_tidak_layak = []

daftar_kamus = [
    f for f in os.listdir(directory_JSON_raw)
    if f.endswith(".json") and not f.startswith(".")
]

daftar_kamus = sorted(daftar_kamus)
for filename in tqdm(daftar_kamus):
    print("====" + filename + "====")
    try:
        with open(directory_JSON_raw + "/" + filename,"rb") as f:
            data = js.load(f)
        f.close()
        
        new_filename = os.path.splitext(filename)[0]
        result_raw = extract_information_json(data)
        result_clean = clean_information(result_raw)
        result = seperate_span(result_clean)
        
        if (is_contain_bold_and_italic(result["font"])):
            csv_res = pd.DataFrame.from_dict(result)
            csv_res.to_csv(directory_hasil + "/" + new_filename + "-ekstraksi.csv",index=False)
            daftar_kamus_layak.append(new_filename)
        
        else:
            daftar_kamus_tidak_layak.append(new_filename)
    except Exception as e:
        print(f"Error occurred while processing {filename}: {e}")
        daftar_kamus_tidak_layak.append(new_filename)
        continue

print("Jumlah kamus yang layak:", len(daftar_kamus_layak))
print("Jumlah kamus yang tidak layak:", len(daftar_kamus_tidak_layak))

  0%|          | 0/88 [00:00<?, ?it/s]

====1. Kamus Makasar-Indonesia (1995)-hasil.json====


  1%|          | 1/88 [00:01<01:38,  1.13s/it]

====10. Kamus Bahasa Indonesia-Dayak Deah Edisi I (2013)-hasil.json====


  2%|▏         | 2/88 [00:02<01:28,  1.03s/it]

...

====92. Kamus Bahasa Haloban (2013)-hasil.json====


 99%|█████████▉| 87/88 [01:38<00:00,  1.32it/s]

====94. Kamus Bahasa Madura-Indonesia (1977)-hasil.json====


100%|██████████| 88/88 [01:40<00:00,  1.14s/it]

Jumlah kamus yang layak: 68
Jumlah kamus yang tidak layak: 20


In [42]:
print("==== Daftar Kamus Layak ====")

for i in daftar_kamus_layak:
    print(i)

==== Daftar Kamus Layak ====
1. Kamus Makasar-Indonesia (1995)-hasil
10. Kamus Bahasa Indonesia-Dayak Deah Edisi I (2013)-hasil
11. Kamus Suwawa-Indonesia (1985)-hasil
12. Kamus Bahasa Indonesia-Kaidipang L-Z (2000)-hasil
13. Kamus Bahasa Indonesia-Bahasa Sunda I (1993)-hasil
14. Kamus Indonesia-Minangkabau II (1994)-hasil
15. Kamus Bahasa Indonesia-Pasir (1997)-hasil
16. Kamus Bahasa Indonesia Karo A-K (1998)-hasil
17. Kamus Melayu Makasar-Indonesia (1985)-hasil
18. Kamus Bahasa Jawa-Bahasa Indonesia I (1993)-hasil
19. Kamus Bahasa Indoensia-Melayu Riau (1997)-hasil
2. Kamus Melayu-Indonesia (1985)-hasil
20. Kamus Bahasa Melayu Ambon-Indonesia (1998)-hasil
21. Kamus Bahasa Indonesia-Sentani A-K (1999)-hasil
22. Kamus Bahasa Gorontalo-Indonesia (1977)-hasil
23. Kamus Dwibahasa Dayak Ngaju-Indonesia (2013)-hasil
24. Kamus Minangkabau-Indonesia (1985)-hasil
25. Kamus Bahasa Indonesia Jambi L-Z (1997)-hasil
26. Kamus Bahasa Indonesia-Bahasa Tonsea II (1996)-hasil
27. Kamus Bahasa Indonesi

In [43]:
print("==== Daftar Kamus Tidak Layak ====")

for i in daftar_kamus_tidak_layak:
    print(i)

==== Daftar Kamus Tidak Layak ====
39. Kamus Alas Indonesia (1985)-hasil
47. Kamus Bahasa Karo-Indonesia (2001)-hasil
53. Kamus Bakumpai Indonesia (1985)-hasil
64. Kamus Culambacu-Indonesia (2018)-hasil
65. Kamus Angkola Mandailing-Indonesia ed 2 (2016)-hasil
69. Kamus Dwi Bahasa Alune-Indonesia (-)-hasil
69. Kamus Dwi Bahasa Alune-Indonesia (-)-hasil
73. Kamus Samawa-Indonesia (2015)-hasil
75. Kamus dwibahasa Indonesia-Madura (2008)-hasil
80. Kamus Melayu Sumatera Utara-Indonesia (2018)-hasil
81. Kamus Bali-Indonesia ed3 (2016)-hasil
86. Kamus Bahasa Indonesia-Minangkabau edrevisi (2013)-hasil
88. Kamus Sasak-Indonesia ed2 (2017)-hasil
88b. Kamus Sasak-Indonesia ed2 (2018)-hasil
88b. Kamus Sasak-Indonesia ed2 (2018)-hasil
89b. Kamus Dwibahasa Bahasa Mooi-Bahasa Indonesia (2017)-hasil
90. Kamus Mbojo-Indonesia (2015)-hasil
90b. Kamus Mbojo-Indonesia (2015)-hasil
91b. Kamus Bahasa Simalungun-Indonesia ed2 (2016)-hasil
92. Kamus Bahasa Haloban (2013)-hasil


In [44]:
split_kamus = pd.read_csv("../Split Kamus.csv")

kamus_split = split_kamus["Kamus"].values.tolist()

awal = split_kamus["halaman_pertama"].values.tolist()

akhir = split_kamus["halaman_terakhir"].values.tolist()

In [45]:
range_lookup = split_kamus.set_index("Kamus").to_dict("index")

In [46]:
range_lookup

{1: {'halaman_pertama': 31, 'halaman_terakhir': 476, 'Unnamed: 3': nan},
 2: {'halaman_pertama': 8, 'halaman_terakhir': 237, 'Unnamed: 3': nan},
 3: {'halaman_pertama': 38, 'halaman_terakhir': 339, 'Unnamed: 3': nan},
 4: {'halaman_pertama': 18, 'halaman_terakhir': 225, 'Unnamed: 3': nan},
 5: {'halaman_pertama': 10, 'halaman_terakhir': 158, 'Unnamed: 3': nan},
 8: {'halaman_pertama': 10, 'halaman_terakhir': 163, 'Unnamed: 3': nan},
 9: {'halaman_pertama': 21, 'halaman_terakhir': 156, 'Unnamed: 3': nan},
 10: {'halaman_pertama': 16, 'halaman_terakhir': 315, 'Unnamed: 3': nan},
 11: {'halaman_pertama': 26, 'halaman_terakhir': 360, 'Unnamed: 3': nan},
 12: {'halaman_pertama': 18, 'halaman_terakhir': 529, 'Unnamed: 3': nan},
 13: {'halaman_pertama': 24, 'halaman_terakhir': 431, 'Unnamed: 3': nan},
 14: {'halaman_pertama': 6, 'halaman_terakhir': 470, 'Unnamed: 3': nan},
 15: {'halaman_pertama': 18, 'halaman_terakhir': 546, 'Unnamed: 3': nan},
 16: {'halaman_pertama': 18, 'halaman_terakhir'

In [47]:
import re

def extract_kamus_number(filename):
    return int(re.match(r"\d+", filename).group())

In [48]:
import os
import pandas as pd
import re

directory_hasil_final = "../[Full] CSV JSON all information - Final"
shutil.rmtree(directory_hasil_final)
os.makedirs(directory_hasil_final)

range_lookup = split_kamus.set_index("Kamus").to_dict("index")

directory_hasil_raw = os.listdir(directory_hasil)

daftar_kamus_tidak_layak = []
daftar_kamus_layak_setelah_displit = []

for filename in directory_hasil_raw:

    print("====", filename, "====")

    kamus_number = extract_kamus_number(filename)

    if kamus_number not in range_lookup:
        print("Range not found for kamus", kamus_number)
        continue

    awal = range_lookup[kamus_number]["halaman_pertama"]
    akhir = range_lookup[kamus_number]["halaman_terakhir"]

    kamus = pd.read_csv(os.path.join(directory_hasil, filename))

    kamus = kamus[kamus["page"] >= awal]
    kamus = kamus[kamus["page"] <= akhir]
    kamus = kamus.reset_index(drop=True)

    if is_contain_bold(kamus["font"].tolist()):
        print("==== Memenuhi Kriteria ====")

        kamus.to_csv(
            os.path.join(directory_hasil_final, filename),
            index=False
        )

        daftar_kamus_layak_setelah_displit.append(filename)

    else:
        print("==== Tidak Memenuhi Kriteria ====")
        daftar_kamus_tidak_layak.append(filename)

==== 1. Kamus Makasar-Indonesia (1995)-hasil-ekstraksi.csv ====
==== Memenuhi Kriteria ====
==== 10. Kamus Bahasa Indonesia-Dayak Deah Edisi I (2013)-hasil-ekstraksi.csv ====
==== Memenuhi Kriteria ====
==== 11. Kamus Suwawa-Indonesia (1985)-hasil-ekstraksi.csv ====
==== Memenuhi Kriteria ====
==== 12. Kamus Bahasa Indonesia-Kaidipang L-Z (2000)-hasil-ekstraksi.csv ====
==== Memenuhi Kriteria ====
==== 13. Kamus Bahasa Indonesia-Bahasa Sunda I (1993)-hasil-ekstraksi.csv ====
==== Memenuhi Kriteria ====
==== 14. Kamus Indonesia-Minangkabau II (1994)-hasil-ekstraksi.csv ====
==== Memenuhi Kriteria ====
==== 15. Kamus Bahasa Indonesia-Pasir (1997)-hasil-ekstraksi.csv ====
==== Memenuhi Kriteria ====
==== 16. Kamus Bahasa Indonesia Karo A-K (1998)-hasil-ekstraksi.csv ====
==== Memenuhi Kriteria ====
==== 17. Kamus Melayu Makasar-Indonesia (1985)-hasil-ekstraksi.csv ====
==== Memenuhi Kriteria ====
==== 18. Kamus Bahasa Jawa-Bahasa Indonesia I (1993)-hasil-ekstraksi.csv ====
==== Memenuhi K

In [49]:
# daftar_kamus_tidak_layak = []
# daftar_kamus_layak_setelah_displit = []

# # directory_hasil_raw = os.listdir(directory_hasil)
# directory_hasil_raw = sorted(os.listdir(directory_hasil))
# for i in range(len(directory_hasil_raw)):
#     filename_clean = directory_hasil_raw[i]
#     print("====" + filename_clean + "====" + str(kamus_split[i]))
#     print("====" + filename_clean + "====")
    
#     kamus = pd.read_csv(directory_hasil + "/" + filename_clean)
#     kamus = kamus[kamus["page"] >= awal[i]]
#     kamus = kamus[kamus["page"] <= akhir[i]]
#     kamus = kamus.reset_index(drop=True)
    
#     if is_contain_bold(kamus["font"].values.tolist()):
#         print("==== Memenuhi Kriteria ====")
#         directory_hasil_clean = "[Full] CSV JSON all information"
#         kamus.to_csv(directory_hasil_clean + "/" + filename_clean,index=False)
#         daftar_kamus_layak_setelah_displit.append(filename_clean)
#     else:
#         print("==== Tidak Memenuhi Kriteria ====")
#         daftar_kamus_tidak_layak.append(filename_clean)
    

In [50]:
print("Jumlah kamus yang tidak layak:", len(daftar_kamus_tidak_layak))
for i in daftar_kamus_tidak_layak:
    print(i)

Jumlah kamus yang tidak layak: 2
82. Kamus dwibahasa Indonesia - Aceh (2011)-hasil-ekstraksi.csv
83. Kamus dwibahasa Bahasa Indonesia - Dayak Halong (2017)-hasil-ekstraksi.csv


In [51]:
print("Jumlah kamus yang layak:", len(daftar_kamus_layak_setelah_displit))
for i in daftar_kamus_layak_setelah_displit:
    print(i)

Jumlah kamus yang layak: 66
1. Kamus Makasar-Indonesia (1995)-hasil-ekstraksi.csv
10. Kamus Bahasa Indonesia-Dayak Deah Edisi I (2013)-hasil-ekstraksi.csv
11. Kamus Suwawa-Indonesia (1985)-hasil-ekstraksi.csv
12. Kamus Bahasa Indonesia-Kaidipang L-Z (2000)-hasil-ekstraksi.csv
13. Kamus Bahasa Indonesia-Bahasa Sunda I (1993)-hasil-ekstraksi.csv
14. Kamus Indonesia-Minangkabau II (1994)-hasil-ekstraksi.csv
15. Kamus Bahasa Indonesia-Pasir (1997)-hasil-ekstraksi.csv
16. Kamus Bahasa Indonesia Karo A-K (1998)-hasil-ekstraksi.csv
17. Kamus Melayu Makasar-Indonesia (1985)-hasil-ekstraksi.csv
18. Kamus Bahasa Jawa-Bahasa Indonesia I (1993)-hasil-ekstraksi.csv
19. Kamus Bahasa Indoensia-Melayu Riau (1997)-hasil-ekstraksi.csv
2. Kamus Melayu-Indonesia (1985)-hasil-ekstraksi.csv
20. Kamus Bahasa Melayu Ambon-Indonesia (1998)-hasil-ekstraksi.csv
21. Kamus Bahasa Indonesia-Sentani A-K (1999)-hasil-ekstraksi.csv
22. Kamus Bahasa Gorontalo-Indonesia (1977)-hasil-ekstraksi.csv
23. Kamus Dwibahasa Day